# Buku Kerja Hari 5 — Visualisasi Data, API & Web Scraping

Pendamping **Modul 5**. Estimasi 6 jam. Prasyarat: Hari 1–4 (`pip install matplotlib seaborn requests beautifulsoup4 feedparser`).

**Tujuan:** membuat grafik yang informatif dengan Matplotlib dan Seaborn, mengambil data dari API dengan `requests` (dengan penanganan error), mengurai HTML dengan BeautifulSoup dan feed Atom arXiv dengan `feedparser`, lalu menggabungkannya dalam mini project.

| Bagian | Isi |
|---|---|
| A | Matplotlib (L1–L5) |
| B | Seaborn (L6–L8) |
| C | API & JSON (L9–L11) |
| D | Scraping & arXiv (L12–L13) |
| E | Mini Project 1–4 (P1–P4) |
| F | Rekap |

> **Offline-friendly:** semua latihan memakai data contoh di dalam notebook (atau server lokal) sehingga bisa dikerjakan tanpa internet. Sel opsional bertanda *internet* memanggil API sungguhan.
>
> **Pola latihan grafik:** tulis function yang **mengembalikan objek figure/axes**; pemeriksa mengecek isinya (judul, label, jumlah batang/garis). Jalankan sel "Lihat hasil" untuk melihat gambarnya.

> **Cara memakai:** jalankan sel dari atas ke bawah (`Shift+Enter`). Pada setiap latihan, ganti baris `# TULIS KODE ANDA DI SINI` dengan jawaban Anda, lalu jalankan sel **Periksa jawaban** di bawahnya. Buka *Petunjuk* bila buntu. Solusi lengkap ada di notebook `solusi/` — buka hanya setelah mencoba sendiri.

In [ ]:
# @title Jalankan sel ini lebih dulu (alat bantu pemeriksa jawaban)
from contextlib import contextmanager
from unittest.mock import patch

STRICT = False
_hasil = {}

@contextmanager
def periksa(kode):
    """Dipakai di sel 'Periksa jawaban': gagal -> pesan petunjuk; lolos -> tanda centang."""
    try:
        yield
        _hasil[kode] = True
        print(f"✅ {kode} benar!")
    except AssertionError as e:
        _hasil[kode] = False
        print(f"❌ {kode} belum tepat: {e}")
        if STRICT: raise
    except Exception as e:
        _hasil[kode] = False
        print(f"❌ {kode} belum bisa dijalankan: {type(e).__name__}: {e}")
        if STRICT: raise

@contextmanager
def masukan(*jawaban):
    """Simulasi input(): with masukan('Ani', '80'): ... -> input() mengembalikan 'Ani' lalu '80'."""
    with patch("builtins.input", side_effect=list(jawaban)):
        yield

def ringkasan(semua):
    """Cetak rekap latihan: ✅ benar, ❌ salah, ⏳ belum dicoba."""
    benar = 0
    for kode in semua:
        s = _hasil.get(kode)
        tanda = "⏳ belum dicoba" if s is None else ("✅ benar" if s else "❌ perlu diperbaiki")
        benar += bool(s)
        print(f"{kode:<6} {tanda}")
    print(f"\nSkor: {benar} dari {len(semua)} latihan benar")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_theme(style="whitegrid")
WORK = Path("latihan_hari5")
WORK.mkdir(exist_ok=True)

# Data contoh (fiktif) mirip dataset 'tips' agar bisa dipakai tanpa internet
_rng = np.random.default_rng(1)
tips_mini = pd.DataFrame({
    "total_bill": _rng.gamma(4, 5, 120).round(2),
    "day": _rng.choice(["Thur", "Fri", "Sat", "Sun"], 120),
    "sex": _rng.choice(["Female", "Male"], 120),
})
tips_mini["tip"] = (tips_mini["total_bill"] * 0.15 + _rng.normal(0, 1, 120)).clip(lower=0.5).round(2)

def tutup():
    plt.close("all")


## A. Matplotlib

**Ringkas:** `fig, ax = plt.subplots()` membuat kanvas; `ax.plot/bar/hist/scatter/pie` menggambar; `ax.set_title/xlabel/ylabel` memberi label; `ax.legend()` memunculkan legenda; `fig.tight_layout()` merapikan. Setiap grafik wajib punya **judul** dan **label sumbu**.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot([1, 2, 3], [2, 4, 3], marker="o", label="contoh")
ax.set_title("Contoh garis"); ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend()
fig.tight_layout(); plt.show()

### Latihan L1 — Line plot dua seri

Buat function `plot_garis(x, seri, judul)` dengan `seri` berupa dict `{nama: list_y}`. Gambar satu garis per seri (beri `label`), pasang `judul`, label sumbu x dan y (tidak kosong), dan legenda. Kembalikan `fig`.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
fig, ax ← plt.subplots()
UNTUK setiap (nama, y) di seri: ax.plot(x, y, label=nama)
set_title, set_xlabel, set_ylabel, legend → RETURN fig
```

</details>

In [ ]:
def plot_garis(x, seri, judul):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('L1'):
    fig = plot_garis(["Jan", "Feb", "Mar"], {"Ani": [75, 80, 85], "Budi": [70, 72, 78]}, "Perkembangan Nilai")
    ax = fig.axes[0]
    assert len(ax.lines) == 2, f"harus 2 garis, ada {len(ax.lines)}"
    assert ax.get_title() == "Perkembangan Nilai", f"judul: {ax.get_title()!r}"
    assert ax.get_xlabel().strip() and ax.get_ylabel().strip(), "label sumbu x dan y tidak boleh kosong"
    assert ax.get_legend() is not None, "tambahkan ax.legend()"
    assert sorted(l.get_label() for l in ax.lines) == ["Ani", "Budi"]
    tutup()

In [ ]:
# Lihat hasil
plot_garis(["Jan", "Feb", "Mar", "Apr"], {"Produk A": [120, 135, 150, 140], "Produk B": [90, 100, 110, 105]}, "Penjualan A vs B");

### Latihan L2 — Bar chart dengan label nilai

Buat `plot_bar(kategori, nilai, judul)`: batang vertikal, tulis angka di atas setiap batang (`ax.text`), beri judul dan label sumbu. Kembalikan `fig`.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
batang ← ax.bar(kategori, nilai)
UNTUK (b, v): ax.text(x_tengah_batang, tinggi + sedikit, str(v))
x_tengah = b.get_x() + b.get_width() / 2
```

</details>

In [ ]:
def plot_bar(kategori, nilai, judul):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('L2'):
    fig = plot_bar(["IF", "SI", "TI"], [120, 95, 80], "Mahasiswa per Prodi")
    ax = fig.axes[0]
    assert len(ax.patches) == 3, f"harus 3 batang, ada {len(ax.patches)}"
    assert [round(p.get_height()) for p in ax.patches] == [120, 95, 80]
    assert len(ax.texts) == 3, f"harus ada 3 teks angka di atas batang, ada {len(ax.texts)}"
    assert ax.get_title() == "Mahasiswa per Prodi" and ax.get_xlabel().strip() and ax.get_ylabel().strip()
    tutup()

In [ ]:
# Lihat hasil
plot_bar(["IF", "SI", "TI", "Manajemen"], [120, 95, 80, 150], "Jumlah Mahasiswa per Prodi");

### Latihan L3 — Histogram dengan garis rata-rata

Buat `plot_hist(data, bins, judul)`: histogram dengan `bins` kotak, garis vertikal merah putus-putus di rata-rata (`ax.axvline`, beri `label` berisi nilai rata-rata), judul, dan label sumbu. Kembalikan `fig`.

In [ ]:
def plot_hist(data, bins, judul):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('L3'):
    data = np.random.default_rng(0).normal(70, 8, 200)
    fig = plot_hist(data, 12, "Distribusi Nilai")
    ax = fig.axes[0]
    kotak = [p for p in ax.patches]
    assert len(kotak) == 12, f"harus 12 kotak, ada {len(kotak)}"
    assert len(ax.lines) == 1, "tambahkan satu garis rata-rata (ax.axvline)"
    assert abs(ax.lines[0].get_xdata()[0] - data.mean()) < 1e-6, "garis harus tepat di rata-rata"
    assert ax.get_title() == "Distribusi Nilai" and ax.get_xlabel().strip() and ax.get_ylabel().strip()
    tutup()

In [ ]:
# Lihat hasil
plot_hist(np.random.default_rng(42).normal(75, 10, 250), 20, "Histogram Nilai");

### Latihan L4 — Dashboard 2×2 dengan subplot

Buat `dashboard()` yang mengembalikan figure 2×2 (`plt.subplots(2, 2)`) berisi: (0,0) garis `[1,4,9,16]`, (0,1) bar `A,B,C → 10,20,15`, (1,0) histogram 200 data acak ber-seed, (1,1) scatter 40 titik acak ber-seed. **Setiap panel punya judul** dan figure punya `suptitle`.

In [ ]:
def dashboard():
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('L4'):
    fig = dashboard()
    assert len(fig.axes) == 4, f"harus 4 panel, ada {len(fig.axes)}"
    assert all(a.get_title().strip() for a in fig.axes), "setiap panel harus punya judul"
    assert fig._suptitle is not None and fig._suptitle.get_text().strip(), "tambahkan fig.suptitle(...)"
    assert len(fig.axes[0].lines) == 1 and len(fig.axes[1].patches) == 3
    tutup()

In [ ]:
# Lihat hasil
dashboard();

### Latihan L5 — Simpan grafik ke file

Buat `simpan_grafik(fig, nama_file)` yang menyimpan `fig` ke `latihan_hari5/<nama_file>` dengan `dpi=120` dan `bbox_inches="tight"`, lalu mengembalikan objek `Path` file tersebut.

In [ ]:
def simpan_grafik(fig, nama_file):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('L5'):
    fig, ax = plt.subplots(); ax.plot([1, 2, 3])
    p = simpan_grafik(fig, "uji.png")
    assert p == WORK / "uji.png", f"path: {p}"
    assert p.exists() and p.stat().st_size > 500, "file gambar belum tersimpan"
    tutup()

## B. Seaborn

**Ringkas:** Seaborn membuat grafik statistik dari DataFrame dengan sedikit kode: `sns.histplot`, `sns.boxplot`, `sns.scatterplot`, `sns.countplot`, `sns.barplot`, `sns.heatmap`. Argumen `data=`, `x=`, `y=`, `hue=`. Latihan memakai `tips_mini` (data contoh yang sudah disediakan di setup).

In [ ]:
print(tips_mini.head(3))
ax = sns.boxplot(data=tips_mini, x="day", y="total_bill")
ax.set_title("Contoh boxplot"); plt.show()

### Latihan L6 — Boxplot per hari

Buat `plot_box(df)` yang menggambar boxplot `total_bill` per `day` (urutan `Thur, Fri, Sat, Sun`) dengan judul `"Total Bill per Hari"`. Kembalikan `ax`.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
sns.boxplot(data=df, x="day", y="total_bill", order=[...])
lalu ax.set_title(...) dan RETURN ax
```

</details>

In [ ]:
def plot_box(df):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('L6'):
    ax = plot_box(tips_mini)
    assert ax.get_title() == "Total Bill per Hari", f"judul: {ax.get_title()!r}"
    label = [t.get_text() for t in ax.get_xticklabels()]
    assert label == ["Thur", "Fri", "Sat", "Sun"], f"urutan sumbu x: {label}"
    assert ax.get_ylabel() == "total_bill"
    tutup()

In [ ]:
# Lihat hasil
plot_box(tips_mini);

### Latihan L7 — Heatmap korelasi

Buat `plot_korelasi(df)` yang menggambar heatmap korelasi kolom **numerik** `df` dengan angka (`annot=True`, `fmt=".2f"`, `cmap="coolwarm"`, `center=0`) dan judul `"Korelasi"`. Kembalikan `ax`.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
korelasi ← df.select_dtypes(include="number").corr()
sns.heatmap(korelasi, annot=True, fmt=".2f", cmap="coolwarm", center=0)
```

</details>

In [ ]:
def plot_korelasi(df):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('L7'):
    ax = plot_korelasi(tips_mini)
    assert ax.get_title() == "Korelasi"
    mesh = ax.collections[0]
    assert mesh.get_array().shape == (2, 2) or mesh.get_array().size == 4, f"heatmap harus 2×2 (total_bill, tip); ukuran {mesh.get_array().shape}"
    assert len(ax.texts) == 4, "aktifkan annot=True agar angka tampil"
    tutup()

### Latihan L8 — Countplot dan barplot

Buat `plot_hitung(df)` yang mengembalikan `ax` berisi **countplot** jumlah transaksi per `day` dengan `hue="sex"` (urutan hari `Thur, Fri, Sat, Sun`), judul `"Jumlah Transaksi per Hari"`.

In [ ]:
def plot_hitung(df):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('L8'):
    ax = plot_hitung(tips_mini)
    assert ax.get_title() == "Jumlah Transaksi per Hari"
    jumlah_batang = sum(len(c) for c in ax.containers)
    assert jumlah_batang == 8, f"harus 4 hari × 2 jenis kelamin = 8 batang, ada {jumlah_batang}"
    assert ax.get_legend() is not None, "gunakan hue='sex' agar ada legenda"
    total = int(sum(b.get_height() for c in ax.containers for b in c))
    assert total == len(tips_mini), f"jumlah semua batang harus {len(tips_mini)}, bukan {total}"
    tutup()

## C. API dan JSON

**Ringkas:** `requests.get(url, params=..., timeout=...)` → objek respons; `resp.raise_for_status()` melempar `HTTPError` untuk kode 4xx/5xx; `resp.json()` mengubah JSON ke dict/list. Tangkap `requests.exceptions.RequestException` agar program tidak crash. `pd.json_normalize` meratakan JSON bertingkat menjadi DataFrame.

In [ ]:
# Contoh struktur JSON bertingkat (mirip API pengguna)
users = [
    {"id": 1, "name": "Leanne", "address": {"city": "Gwenborough"}, "company": {"name": "Romaguera"}},
    {"id": 2, "name": "Ervin", "address": {"city": "Wisokyburgh"}, "company": {"name": "Deckow"}},
]
print(pd.json_normalize(users)[["id", "name", "address.city", "company.name"]])

### Latihan L9 — ambil_json yang tahan error

Buat `ambil_json(url, params=None, timeout=10)`: lakukan GET, `raise_for_status()`, lalu kembalikan `resp.json()`. Jika terjadi `requests.exceptions.RequestException` **atau** respons bukan JSON (`ValueError`), cetak `"Gagal: ..."` dan kembalikan `None`. Pemeriksa menjalankan server HTTP lokal sungguhan.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
COBA: resp ← requests.get(url, params, timeout); resp.raise_for_status(); RETURN resp.json()
TANGKAP RequestException atau ValueError: cetak "Gagal", RETURN None
```

</details>

In [ ]:
import requests

def ambil_json(url, params=None, timeout=10):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('L9'):
    import json, threading, time
    from http.server import BaseHTTPRequestHandler, HTTPServer

    class _H(BaseHTTPRequestHandler):
        def log_message(self, *a): pass
        def do_GET(self):
            if self.path.startswith("/ok"):
                body, code, ctype = json.dumps({"halo": "dunia"}), 200, "application/json"
            elif self.path == "/html":
                body, code, ctype = "<html>bukan json</html>", 200, "text/html"
            elif self.path == "/lambat":
                time.sleep(1.0); body, code, ctype = "{}", 200, "application/json"
            else:
                body, code, ctype = "tidak ada", 404, "text/plain"
            self.send_response(code); self.send_header("Content-Type", ctype); self.end_headers()
            try: self.wfile.write(body.encode())
            except BrokenPipeError: pass

    _srv = HTTPServer(("127.0.0.1", 0), _H)
    threading.Thread(target=_srv.serve_forever, daemon=True).start()
    base = f"http://127.0.0.1:{_srv.server_address[1]}"
    try:
        assert ambil_json(base + "/ok") == {"halo": "dunia"}, "respons sukses harus dikembalikan sebagai dict"
        assert ambil_json(base + "/tidak-ada") is None, "HTTP 404 → None"
        assert ambil_json(base + "/html") is None, "bukan JSON → None"
        assert ambil_json(base + "/lambat", timeout=0.2) is None, "timeout → None"
        assert ambil_json("http://127.0.0.1:9/", timeout=1) is None, "server mati → None"
    finally:
        _srv.shutdown()

### Latihan L10 — json_normalize dan agregasi

Dari `users` (JSON bertingkat), buat `df` dengan `pd.json_normalize`, lalu isi `jumlah_per_kota` (Series jumlah user per `address.city`, urut nama kota menaik).

In [ ]:
users = [
    {"id": 1, "name": "A", "address": {"city": "Bandung"}},
    {"id": 2, "name": "B", "address": {"city": "Jakarta"}},
    {"id": 3, "name": "C", "address": {"city": "Bandung"}},
    {"id": 4, "name": "D", "address": {"city": "Surabaya"}},
]
# TULIS KODE ANDA DI SINI
df = ...
jumlah_per_kota = ...

In [ ]:
# Periksa jawaban
with periksa('L10'):
    assert "address.city" in df.columns, f"kolom: {df.columns.tolist()}"
    assert jumlah_per_kota.to_dict() == {"Bandung": 2, "Jakarta": 1, "Surabaya": 1}, f"jumlah_per_kota: {jumlah_per_kota.to_dict()}"
    assert jumlah_per_kota.index.tolist() == ["Bandung", "Jakarta", "Surabaya"]

### Latihan L11 — Mengolah respons cuaca

`respons` meniru JSON Open-Meteo (data **contoh**). Buat `df_cuaca` dengan kolom `tanggal`, `tmax`, `tmin` dari `respons["daily"]`, lalu `hari_terpanas` = tanggal dengan `tmax` tertinggi dan `rentang_rata` = rata-rata (`tmax - tmin`).

In [ ]:
respons = {"daily": {
    "time": ["2026-10-01", "2026-10-02", "2026-10-03"],
    "temperature_2m_max": [32.1, 33.4, 31.0],
    "temperature_2m_min": [24.5, 25.0, 24.1],
}}
# TULIS KODE ANDA DI SINI
df_cuaca = ...
hari_terpanas = ...
rentang_rata = ...

In [ ]:
# Periksa jawaban
with periksa('L11'):
    assert df_cuaca.columns.tolist() == ["tanggal", "tmax", "tmin"], f"kolom: {df_cuaca.columns.tolist()}"
    assert len(df_cuaca) == 3
    assert hari_terpanas == "2026-10-02", f"hari_terpanas: {hari_terpanas}"
    assert abs(rentang_rata - ((32.1-24.5) + (33.4-25.0) + (31.0-24.1)) / 3) < 1e-9, f"rentang_rata: {rentang_rata}"

In [ ]:
# (opsional — INTERNET) Panggilan API sungguhan; dilewati otomatis bila gagal
try:
    r = requests.get("https://api.open-meteo.com/v1/forecast",
                     params={"latitude": -6.2, "longitude": 106.8, "current_weather": True}, timeout=10)
    r.raise_for_status()
    print(r.json()["current_weather"])
except Exception as e:
    print("Lewati (tidak ada internet / API tidak terjangkau):", type(e).__name__)

## D. Scraping dan arXiv

**Etika:** patuhi `robots.txt` dan ketentuan situs, beri jeda antar request, jangan membebani server, jangan mengambil data pribadi. **BeautifulSoup:** `soup.find(...)` bisa mengembalikan `None` — selalu periksa sebelum `.get_text()`. **arXiv:** API mengembalikan feed Atom; `feedparser.parse(teks)` mengurainya.

### Latihan L12 — Scraping HTML kutipan

Dari string `html`, buat `ambil_kutipan(html)` → list dict `{"penulis": ..., "teks": ...}` untuk setiap `<div class="quote">`. Jika elemen `span.text` atau `small.author` tidak ada pada suatu kutipan, **lewati** kutipan itu (jangan crash).

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
untuk setiap div.quote:
    teks ← cari span.text ; penulis ← cari small.author
    JIKA keduanya ada → tambahkan dict ; SELAIN ITU lewati
```

</details>

In [ ]:
from bs4 import BeautifulSoup

html = '''
<div class="quote"><span class="text">Imajinasi lebih penting dari pengetahuan.</span><small class="author">Einstein</small></div>
<div class="quote"><span class="text">Tanpa penulis.</span></div>
<div class="quote"><span class="text">Jadilah dirimu sendiri.</span><small class="author">Wilde</small></div>
'''

def ambil_kutipan(html):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('L12'):
    h = ambil_kutipan(html)
    assert [x["penulis"] for x in h] == ["Einstein", "Wilde"], f"hasil: {h}"
    assert h[0]["teks"] == "Imajinasi lebih penting dari pengetahuan."
    assert ambil_kutipan("<p>kosong</p>") == []

### Latihan L13 — Mengurai feed arXiv

`xml` adalah feed Atom contoh (data **fiktif**). Buat `baca_feed(xml)` → DataFrame kolom `judul`, `penulis` (gabungan nama dipisah `", "`), `tahun` (int dari `published`). Buat juga `kata_sering(df, n)` → list `(kata, jumlah)` n kata judul teratas (huruf kecil, ≥ 3 huruf, tanpa stopword `{"the","for","and","of","in","with"}`).

In [ ]:
import feedparser
from collections import Counter
import re

xml = '''<?xml version="1.0"?>
<feed xmlns="http://www.w3.org/2005/Atom">
 <entry><title>Deep Learning for Education</title><published>2024-03-01T00:00:00Z</published>
   <author><name>A. Contoh</name></author><author><name>B. Sampel</name></author></entry>
 <entry><title>Learning Analytics with Deep Models</title><published>2025-06-10T00:00:00Z</published>
   <author><name>C. Fiktif</name></author></entry>
 <entry><title>Deep Learning in Practice</title><published>2025-09-02T00:00:00Z</published>
   <author><name>A. Contoh</name></author></entry>
</feed>'''

def baca_feed(xml):
    # TULIS KODE ANDA DI SINI
    pass

def kata_sering(df, n):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('L13'):
    df = baca_feed(xml)
    assert df.columns.tolist() == ["judul", "penulis", "tahun"], f"kolom: {df.columns.tolist()}"
    assert len(df) == 3 and df["tahun"].tolist() == [2024, 2025, 2025]
    assert df["penulis"].iloc[0] == "A. Contoh, B. Sampel"
    top = kata_sering(df, 2)
    assert top[0] == ("deep", 3) and top[1] == ("learning", 3), f"top: {top}"
    assert kata_sering(df, 100)[-1][1] >= 1

## E. Mini Project

### Latihan P1 — Project 1 — Dashboard Visualisasi Nilai (Dasar)

Buat `dashboard_nilai(df)` untuk DataFrame `nama, prodi, nilai` yang mengembalikan figure 2×2: (0,0) histogram `nilai`, (0,1) bar rata-rata `nilai` per `prodi`, (1,0) pie proporsi jumlah mahasiswa per prodi, (1,1) boxplot `nilai` per `prodi`. Setiap panel punya judul tidak kosong; figure punya `suptitle`.

In [ ]:
def dashboard_nilai(df):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('P1'):
    d = pd.DataFrame({"nama": list("ABCDEFGH"), "prodi": ["IF", "SI"] * 4, "nilai": [80, 70, 90, 65, 85, 75, 95, 60]})
    fig = dashboard_nilai(d)
    assert len(fig.axes) == 4, f"harus 4 panel, ada {len(fig.axes)}"
    assert all(a.get_title().strip() for a in fig.axes), "setiap panel harus punya judul"
    assert fig._suptitle is not None and fig._suptitle.get_text().strip()
    bar = fig.axes[1]
    assert sorted(round(p.get_height(), 2) for p in bar.patches) == [67.5, 87.5], f"tinggi batang: {[p.get_height() for p in bar.patches]}"
    assert len(fig.axes[2].patches) == 2, "pie harus 2 irisan (IF, SI)"
    tutup()

In [ ]:
# Lihat hasil
d = pd.DataFrame({"nama": list("ABCDEFGH"), "prodi": ["IF", "SI"] * 4, "nilai": [80, 70, 90, 65, 85, 75, 95, 60]})
dashboard_nilai(d);

### Latihan P2 — Project 2 — API + Visualisasi (Menengah)

Data `posts` (list of dict, meniru JSONPlaceholder; **contoh**). Buat `jumlah_post_per_user(posts)` → Series (indeks `userId`, urut menaik) jumlah post per user, dan `plot_jumlah_post(seri)` → `ax` bar chart dengan judul `"Jumlah Post per User"` dan label sumbu.

In [ ]:
posts = [{"userId": u, "id": i, "title": f"judul {i}"} for i, u in enumerate([1, 1, 1, 2, 2, 3, 3, 3, 3, 4], 1)]

def jumlah_post_per_user(posts):
    # TULIS KODE ANDA DI SINI
    pass

def plot_jumlah_post(seri):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('P2'):
    s = jumlah_post_per_user(posts)
    assert s.to_dict() == {1: 3, 2: 2, 3: 4, 4: 1}, f"seri: {s.to_dict()}"
    assert s.index.tolist() == [1, 2, 3, 4]
    ax = plot_jumlah_post(s)
    assert ax.get_title() == "Jumlah Post per User" and ax.get_xlabel().strip() and ax.get_ylabel().strip()
    assert [int(p.get_height()) for p in ax.patches] == [3, 2, 4, 1]
    tutup()

### Latihan P3 — Project 3 — Pipeline Paper arXiv (Menengah–Tinggi)

Gunakan `baca_feed` dan `kata_sering` dari L13. Buat `analisis_arxiv(xml)` yang mengembalikan dict: `per_tahun` (Series jumlah paper per tahun, urut tahun), `kata` (`kata_sering(df, 3)`), dan `penulis_teratas` (nama penulis dengan paper terbanyak, string; pecah kolom `penulis` pada `", "`). Gunakan feed `xml` dari L13.

In [ ]:
def analisis_arxiv(xml):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('P3'):
    r = analisis_arxiv(xml)
    assert r["per_tahun"].to_dict() == {2024: 1, 2025: 2}, f"per_tahun: {r['per_tahun'].to_dict()}"
    assert len(r["kata"]) == 3 and r["kata"][0][0] == "deep"
    assert r["penulis_teratas"] == "A. Contoh", f"penulis_teratas: {r['penulis_teratas']}"

### Latihan P4 — Project 4 — EDA Lengkap + Visual (Tinggi)

Buat `figur_eda(df)` untuk `tips_mini` (atau data serupa) yang mengembalikan figure 2×2: (0,0) histogram `total_bill`, (0,1) bar rata-rata `tip` per `day`, (1,0) scatter `total_bill` vs `tip` (warna per `sex`), (1,1) heatmap korelasi numerik. Semua panel berjudul; figure punya `suptitle`. Tambahkan juga `insight(df)` → list **minimal 3 string** kalimat insight yang memuat angka dari data (mis. rata-rata tip, hari dengan tip terbesar).

In [ ]:
def figur_eda(df):
    # TULIS KODE ANDA DI SINI
    pass

def insight(df):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('P4'):
    fig = figur_eda(tips_mini)
    assert len(fig.axes) >= 4, f"minimal 4 panel, ada {len(fig.axes)}"
    panel = [a for a in fig.axes if a.get_title().strip()]
    assert len(panel) >= 4, "setiap panel utama harus berjudul"
    assert fig._suptitle is not None and fig._suptitle.get_text().strip()
    i = insight(tips_mini)
    assert isinstance(i, list) and len(i) >= 3 and all(isinstance(x, str) for x in i), "insight harus list minimal 3 string"
    assert any(any(ch.isdigit() for ch in x) for x in i), "insight harus memuat angka dari data"
    tutup()

In [ ]:
# Lihat hasil
figur_eda(tips_mini);

## F. Rekap

1. Pilih satu grafik buatan Anda: pertanyaan apa yang dijawabnya? Jika tidak ada, apakah grafik itu perlu?
2. Mengapa API call harus punya `timeout` dan penanganan error? Apa yang terjadi bila tidak ada?
3. Apa risiko scraping tanpa memeriksa ketentuan situs dan `robots.txt`?

> *Jawaban Anda:* …

**Checklist sebelum Hari 6 (Capstone)**

- [ ] Semua latihan L1–L13 berstatus ✅
- [ ] Project P1–P2 lulus (P3–P4 sebagai tantangan)
- [ ] Setiap grafik saya punya judul, label sumbu, dan alasan keberadaan
- [ ] Saya tahu cara menyimpan data mentah sebelum memprosesnya
- [ ] Saya bisa menulis `try/except` untuk panggilan API

## Rekap hasil latihan

Jalankan sel di bawah setelah mengerjakan latihan.

In [ ]:
ringkasan(['L1', 'L2', 'L3', 'L4', 'L5', 'L6', 'L7', 'L8', 'L9', 'L10', 'L11', 'L12', 'L13', 'P1', 'P2', 'P3', 'P4'])